#  Multi-label Classification

**다중 분류 vs 다중 레이블 분류**

| 구분    | 다중 분류 (Multi-class)          | 다중 레이블 분류 (Multi-label)         |
| ----- | ---------------------------- | ------------------------------- |
| 정의    | 하나의 샘플이 여러 클래스 중 **하나**에만 속함 | 하나의 샘플이 **여러 클래스에 동시에** 속할 수 있음 |
| 예시    | 고양이, 개, 새 중 하나               | 영화가 Action + Sci-Fi + Drama     |
| 출력 형태 | 정수 인덱스 (`y=3`)               | 이진 벡터 (`y=[1, 0, 1, 0, 1]`)     |
| 모델 출력 | `argmax` 사용                  | `sigmoid` 후 **각 클래스마다 이진 판단**   |

**다중 레이블 문제의 대표 예시**

* 텍스트 분류 (뉴스 → 여러 주제)
* 영화/음악 장르 분류
* 이미지에서 객체 감지 (여러 객체 포함 가능)
* 질병 진단 (동시 복합 질병)

## MultiLabelBinarizer

In [3]:
import pandas as pd  # 데이터프레임 처리

data = pd.DataFrame({  # 영화 줄거리와 다중 장르 라벨을 가진 데이터 생성
    'plot': [
        "A man fights crime in a futuristic city.",        # 영화 1 줄거리
        "A love story set in wartime.",                    # 영화 2 줄거리
        "Aliens invade Earth and a war begins.",           # 영화 3 줄거리
        "A detective solves a complicated crime case.",    # 영화 4 줄거리
        "A dramatic romance in the midst of a tragedy."    # 영화 5 줄거리
    ],
    'genres': [
        ['Action', 'Sci-Fi'],              # 영화 1 장르(다중 라벨)
        ['Romance', 'Drama'],              # 영화 2 장르(다중 라벨)
        ['Action', 'Sci-Fi', 'War'],       # 영화 3 장르(다중 라벨)
        ['Crime', 'Mystery'],              # 영화 4 장르(다중 라벨)
        ['Drama', 'Romance']               # 영화 5 장르(다중 라벨)
    ]
})
data  # 데이터 확인

,plot,genres
0,A man fights crime in a futuristic city.,"[Action, Sci-Fi]"
1,A love story set in wartime.,"[Romance, Drama]"
2,Aliens invade Earth and a war begins.,"[Action, Sci-Fi, War]"
3,A detective solves a complicated crime case.,"[Crime, Mystery]"
4,A dramatic romance in the midst of a tragedy.,"[Drama, Romance]"


In [4]:
data.info

<bound method DataFrame.info of                                             plot                 genres
0       A man fights crime in a futuristic city.       [Action, Sci-Fi]
1                   A love story set in wartime.       [Romance, Drama]
2          Aliens invade Earth and a war begins.  [Action, Sci-Fi, War]
3   A detective solves a complicated crime case.       [Crime, Mystery]
4  A dramatic romance in the midst of a tragedy.       [Drama, Romance]>

In [5]:
from sklearn.preprocessing import MultiLabelBinarizer


mlb = MultiLabelBinarizer()
y = mlb.fit_transform(data['genres'])
print(y)
print(mlb.classes_)

label_df = pd.DataFrame(y,columns=mlb.classes_, index=data['plot'])
label_df

[[1 0 0 0 0 1 0]
 [0 0 1 0 1 0 0]
 [1 0 0 0 0 1 1]
 [0 1 0 1 0 0 0]
 [0 0 1 0 1 0 0]]
['Action' 'Crime' 'Drama' 'Mystery' 'Romance' 'Sci-Fi' 'War']


,Action,Crime,Drama,Mystery,Romance,Sci-Fi,War
plot,,,,,,,
A man fights crime in a futuristic city.,1,0,0,0,0,1,0
A love story set in wartime.,0,0,1,0,1,0,0
Aliens invade Earth and a war begins.,1,0,0,0,0,1,1
A detective solves a complicated crime case.,0,1,0,1,0,0,0
A dramatic romance in the midst of a tragedy.,0,0,1,0,1,0,0


## 다중레이블 분류 모델

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer # 텍스트 -> TF-IDF 벡터 변환 도구

vectorizer = TfidfVectorizer()  
X = vectorizer.fit_transform(data['plot'])  # 줄거리 텍스트를 TF-IDF 희소행렬로 변환
input_df = pd.DataFrame(
    X.toarray(),    # 밀집배열 형태로 데이터로 사용()
    columns= vectorizer.get_feature_names_out(),    # Featur 이름
    index = data['plot']
)

input_df

,aliens,and,begins,case,city,complicated,crime,detective,dramatic,earth,...,midst,of,romance,set,solves,story,the,tragedy,war,wartime
plot,,,,,,,,,,,,,,,,,,,,,
A man fights crime in a futuristic city.,0.000000,0.000000,0.000000,0.000000,0.442832,0.000000,0.357274,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
A love story set in wartime.,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.474125,0.000000,0.474125,0.000000,0.000000,0.000000,0.474125
Aliens invade Earth and a war begins.,0.408248,0.408248,0.408248,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.408248,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.408248,0.000000
A detective solves a complicated crime case.,0.000000,0.000000,0.000000,0.463693,0.000000,0.463693,0.374105,0.463693,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.463693,0.000000,0.000000,0.000000,0.000000,0.000000
A dramatic romance in the midst of a tragedy.,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.393795,0.000000,...,0.393795,0.393795,0.393795,0.000000,0.000000,0.000000,0.393795,0.393795,0.000000,0.000000


In [7]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

clf = OneVsRestClassifier(LogisticRegression())
clf.fit(X,y)

print("학습 완료")

학습 완료


In [8]:
test_plot = ['An alien spaceship lands in the middle of a war.']

X_test = vectorizer.transform(test_plot)    # TF-IDF 학습기준으로 벡터화
y_pred = clf.predict(X_test)                # 멀티라벨 예측 (기본 임계값 0.5기준)
print(y_pred)

y_pred_proba = clf.predict_proba(X_test)    # 장르별 예측 확률
print(y_pred_proba)

y_pred = (y_pred_proba >= 0.3).astype(int)  # 임계값 조정(0.3)
print(y_pred)

y_pred_label = mlb.inverse_transform(y_pred)    # 멀티핫 -> 장르 라벨 리스트 역변환
y_pred_label

[[0 0 0 0 0 0 0]]
[[0.38623748 0.17121135 0.44411123 0.17121135 0.44411123 0.38623748
  0.20204508]]
[[1 0 1 0 1 1 0]]


[('Action', 'Drama', 'Romance', 'Sci-Fi')]

## RNN 기반 다중레이블 분류

In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer   # 텍스트 -> 정수 시퀀스형태로 변환
from tensorflow.keras.preprocessing.sequence import pad_sequences   # 시퀀스 길이 패딩/자르기
import torch

tokenizer = Tokenizer(oov_token = 'OOV')    # 사전에 없는단어 OOV
tokenizer.fit_on_texts(data['plot'])        # 줄거리ㄱ로 단어 사전 학습(생성)
X = tokenizer.texts_to_sequences(data['plot'])  # 줄거리를 정수 시퀀스형태로 변환
X = pad_sequences(X,maxlen =10)             # 길이 10으로 패딩처리
X = torch.tensor(X, dtype=torch.long)       # 임베딩 입력용 LongTensor로 변환
X

tensor([[ 0,  0,  2,  5,  6,  4,  3,  2,  7,  8],
        [ 0,  0,  0,  0,  2,  9, 10, 11,  3, 12],
        [ 0,  0,  0, 13, 14, 15, 16,  2, 17, 18],
        [ 0,  0,  0,  2, 19, 20,  2, 21,  4, 22],
        [ 0,  2, 23, 24,  3, 25, 26, 27,  2, 28]])

In [10]:
mlb = MultiLabelBinarizer()             # 다중라벨 -> 멀티핫 변환기
y = mlb.fit_transform(data['genres'])   # 장르 리스트 -> 멀티핫 행렬
y = torch.tensor(y, dtype=torch.float)  # BCE 계열 손실 입력값 -> FloatTensor
y

tensor([[1., 0., 0., 0., 0., 1., 0.],
        [0., 0., 1., 0., 1., 0., 0.],
        [1., 0., 0., 0., 0., 1., 1.],
        [0., 1., 0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 1., 0., 0.]])

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# GRU 분류 모델 : 임베딩 -> GRU -> FC 로 각 라벨별 점수(logit)을 출력하는 다중 라벨 분류 모델
class MultiLabelNet(nn.Module):
    def __init__(self, vocab_size,embedding_dim,hidden_dim,ouput_dim):
        super().__init__()  # nn.Module 초기화
        self.embedding = nn.Embedding(  # 임베딩 (0은 학습영향 최소화)
            vocab_size, 
            embedding_dim, 
            padding_idx = 0
        )
        self.gru = nn.GRU(      # GRU 입력 : 임베딩 -> 출력 : 은닉상태
            embedding_dim,
            hidden_dim,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim,ouput_dim)   # 마지막 은닉상태 -> 라벨 수만큼 logit

    def forward(self,x):
        x = self.embedding(x)           # (B,T) -> (B,T,E)
        _,hidden = self.gru(x)     # output : (B,T,E), hidden : (L,B,H)
        output = self.fc(hidden[-1])    # 입력 : 마지막 레이어의 마지막 은닉상태 (B,H) ->  출력 : (B,1)
        return output   # sigmoid 적용 전 원시 logit
    
vocab_size = len(tokenizer.word_index)+1 # 단어사전 크기(padding index 0 포함)

model = MultiLabelNet(vocab_size, embedding_dim = 100, hidden_dim = 64,ouput_dim=len(mlb.classes_))
model

MultiLabelNet(
  (embedding): Embedding(29, 100, padding_idx=0)
  (gru): GRU(100, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=7, bias=True)
)

In [12]:
from tqdm.auto import tqdm

criterion = nn.BCEWithLogitsLoss()  # 라벨별 이진분류 (내부 sigmoid 포함)
optimizer = optim.Adam(model.parameters(),lr = 0.001)

epochs = 100

for epoch in tqdm(range(epochs)):
    model.train()               # 학습모드

    optimizer.zero_grad()           # 이전 기울기 초기화
    output = model(X)          # 순전파 : 라벨별 로짓 출력
    loss = criterion(output, y)    # 손실 계산
    loss.backward()     # 역전파 : 기울기 계산
    optimizer.step()    # 파라미터 업데이트
    if (epoch+1) %10 ==0:
        print(f"Epoch{epoch+1}/{epochs} Loss : {loss.item():.4f}")

  0%|          | 0/100 [00:00<?, ?it/s]

Epoch10/100 Loss : 0.4778
Epoch20/100 Loss : 0.3154
Epoch30/100 Loss : 0.2081
Epoch40/100 Loss : 0.1439
Epoch50/100 Loss : 0.1058
Epoch60/100 Loss : 0.0817
Epoch70/100 Loss : 0.0657
Epoch80/100 Loss : 0.0544
Epoch90/100 Loss : 0.0461
Epoch100/100 Loss : 0.0398


In [13]:
test_plot = ['An alien spaceship lands in the middle of a war.']

X_test = tokenizer.texts_to_sequences(test_plot)    # 정수 시퀀스 변환
X_test = pad_sequences(X_test, maxlen=10)           # 학습과 동일한 maxlen으로 패딩
X_test = torch.tensor(X_test,dtype=torch.long)      # 임베딩 학습용 LongTensor

model.eval()
with torch.no_grad():
    output = model(X_test)
    p = torch.sigmoid(output)
    print(p)
    pred = (p>=0.5).int()
    pred_label = mlb.inverse_transform(pred)    # 멀티핫 -> 라벨 리스트
    print(pred_label)

tensor([[0.5098, 0.0279, 0.4975, 0.0221, 0.5152, 0.7106, 0.1436]])
[('Action', 'Romance', 'Sci-Fi')]


## Bert Tokenizer / Embedding 적용

In [14]:
%pip install transformers huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [15]:
from transformers import BertTokenizer, BertModel

model_name = 'bert-base-uncased'    # 소문자 영어 기반 Bert 이름
bert_tokenizer = BertTokenizer.from_pretrained(model_name)  # 사전학습 토크나이저
bert_model = BertModel.from_pretrained(model_name)          # 사전학습 Bert 모델 (인코더)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# BERT 임베딩 사전학습 차원 수 : 768
import torch

# 여러 문장을 BERT에 넣어 토큰 단위 문맥 임베딩을 반환
def get_bert_embedding(plots):
    # 토큰화 + 패딩 + 텐서 변환
    encoded = bert_tokenizer(plots,padding=True,truncation=True,return_tensors= 'pt')
    # 임베딩 추출만 할 것이므로 기울기 계산 비활성화
    with torch.no_grad():
        output = bert_model(**encoded)  # encoded : input_ids/token_type_ids/attention_mask

    return output.last_hidden_state     # (batch_size, seq_len, 768) 토큰별 임베딩 반환

plots = data['plot'].values.tolist()    # 줄거리 리스트 
X_tensor = get_bert_embedding(plots)    # Bert 임베딩 생성
print(X_tensor.shape)   # (batch_size, seq_len, embedding_dim)

torch.Size([5, 12, 768])


In [19]:
mlb = MultiLabelBinarizer()             # 다중라벨 -> 멀티핫 변환기
y = mlb.fit_transform(data['genres'])   # 장르 리스트 -> 멀티핫 행렬
y_tensor = torch.tensor(y, dtype=torch.float)  # BCE 계열 손실 입력값 -> FloatTensor
y_tensor

tensor([[1., 0., 0., 0., 0., 1., 0.],
        [0., 0., 1., 0., 1., 0., 0.],
        [1., 0., 0., 0., 0., 1., 1.],
        [0., 1., 0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 1., 0., 0.]])

In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# GRU 분류 모델 : GRU -> FC 로 각 라벨별 점수(logit)을 출력하는 다중 라벨 분류 모델
class MultiLabelNet(nn.Module):
    def __init__(self,input_dim,hidden_dim,output_dim):
        super().__init__()  # nn.Module 초기화
        self.gru = nn.GRU(      # GRU 입력 : 임베딩 -> 출력 : 은닉상태
            input_dim,
            hidden_dim,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim,output_dim)   # 마지막 은닉상태 -> 라벨 수만큼 logit

    def forward(self,x):
        _,hidden = self.gru(x)     # output : (B,T,E), hidden : (L,B,H)
        output = self.fc(hidden[-1])    # 입력 : 마지막 레이어의 마지막 은닉상태 (B,H) ->  출력 : (B,1)
        return output   # sigmoid 적용 전 원시 logit

input_dim = X_tensor.shape[-1]
hidden_dim = 64
output_dim = y_tensor.shape[-1]

model = MultiLabelNet(input_dim,hidden_dim,output_dim)
model

MultiLabelNet(
  (gru): GRU(768, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=7, bias=True)
)

In [31]:
from tqdm.auto import tqdm

criterion = nn.BCEWithLogitsLoss()  # 라벨별 이진분류 (내부 sigmoid 포함)
optimizer = optim.Adam(model.parameters(),lr = 0.001)

epochs = 100

for epoch in tqdm(range(epochs)):
    model.train()               # 학습모드

    optimizer.zero_grad()           # 이전 기울기 초기화
    output = model(X_tensor)          # 순전파 : 라벨별 로짓 출력
    loss = criterion(output, y_tensor)    # 손실 계산
    loss.backward()     # 역전파 : 기울기 계산
    optimizer.step()    # 파라미터 업데이트
    if (epoch+1) %10 ==0:
        print(f"Epoch{epoch+1}/{epochs} Loss : {loss.item():.4f}")

  0%|          | 0/100 [00:00<?, ?it/s]

Epoch10/100 Loss : 0.0007
Epoch20/100 Loss : 0.0006
Epoch30/100 Loss : 0.0004
Epoch40/100 Loss : 0.0004
Epoch50/100 Loss : 0.0003
Epoch60/100 Loss : 0.0003
Epoch70/100 Loss : 0.0002
Epoch80/100 Loss : 0.0002
Epoch90/100 Loss : 0.0002
Epoch100/100 Loss : 0.0002


In [32]:
test_plot = [
    "An alien spaceship lands in the middle of a war.",        # Action, Sci-Fi, War 예상
    "A young couple falls in love during a tragic event.",     # Romance, Drama 예상
    "A detective investigates a mysterious murder case."       # Crime, Mystery 예상
]

X_test = get_bert_embedding(test_plot)


model.eval()
with torch.no_grad():
    output = model(X_test)
    p = torch.sigmoid(output)
    pred = (p>=0.5).int()
    pred_label = mlb.inverse_transform(pred)    # 멀티핫 -> 라벨 리스트
    print(pred_label)

[('Action', 'Sci-Fi', 'War'), ('Action', 'Sci-Fi'), ('Crime', 'Mystery')]
